In [11]:
# Init

import os, json
import librosa
import time
from pathlib import Path
import torch
import numpy as np
from jupyterlab.labextensions import list_flags
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM,
    Wav2Vec2Processor, Wav2Vec2Model, Wav2Vec2FeatureExtractor,
)
from sklearn.cluster import AgglomerativeClustering
import torch.nn.functional as F

os.environ["HF_HUB_OFFLINE"] = "false"
os.environ["TRANSFORMERS_OFFLINE"] = "false"

TEST_DIR = "dataset/test_public"
OUTPUT = "answers.json"

device = "cuda" if torch.cuda.is_available() else "cpu"

with open(os.path.join(TEST_DIR, "prefix.json")) as f:
    prefixes = json.load(f)

whisper_processor = WhisperProcessor.from_pretrained(
    "models/whisper-small",
    local_files_only=True,
)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "models/whisper-small",
    dtype=torch.float16 if device == "cuda" else torch.float32,
    local_files_only=True,
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(
    "models/qwen2.5-0.5b",
    local_files_only=True
)

llm_model = AutoModelForCausalLM.from_pretrained(
    "models/qwen2.5-0.5b",
    dtype=torch.float16 if device == "cuda" else torch.float32,
    local_files_only=True
).to(device).eval()

dialogue_ids = sorted(
    (d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))),
    key=lambda x: int(x) if x.isdigit() else x,
)

WAV2VEC_PATH = "models/wav2vec2-base-960h"
wav2vec_processor = Wav2Vec2Processor.from_pretrained(WAV2VEC_PATH)

wav2vec_model = Wav2Vec2Model.from_pretrained(WAV2VEC_PATH, local_files_only=True).to(device).eval()

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAV2VEC_PATH, local_files_only=True)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: models/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# Audio

def trans_audio(audio_path):
    audio, _ = librosa.load(audio_path, sr=16000)

    features = whisper_processor(audio, sampling_rate=16000, return_tensors='pt').input_features.to(device, dtype=next(whisper_model.parameters()).dtype)

    with torch.inference_mode():
        ids = whisper_model.generate(features, language="en", task="transcribe", max_new_tokens=128)

    return whisper_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

In [13]:
# Splitting Persons

def voice_features(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    return np.concatenate([mfcc.mean(axis=-1), mfcc.std(axis=-1)])

def identify_speakers(files, first, second, iteration=20):
    chunk_ids = [
        int(file.stem.split("_")[1])
        for file in files
    ]

    features = np.stack([
        voice_features(file)
        for file in files
    ])

    first_index = chunk_ids.index(first)
    second_index = chunk_ids.index(second)

    n = len(files)
    speaker_a_count = (n + 1) // 2

    centers = features[[first_index, second_index]].copy()
    free = [
        i for i in range(n)
        if i not in (first_index, second_index)
    ]

    previous = None

    for _ in range(iteration):
        distances = ((features[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)

        ranked = sorted(free, key=lambda i: (distances[i, 0] - distances[i, 1], chunk_ids[i],),)

        labels = np.ones(n, dtype=int)
        labels[first_index] = 0
        labels[ranked[:speaker_a_count - 1]] = 0
        labels[second_index] = 1

        if previous is not None and np.array_equal(labels, previous):
            break

        previous = labels.copy()
        centers = np.stack([features[labels == speaker].mean(axis=0) for speaker in (0, 1)])

    return {
        chunk_id: int(label)
        for chunk_id, label in zip(chunk_ids, labels)
    }

In [14]:
# Helper for LLM-Model

def continuation_logprob(prefix, continuation):
    prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
    continuation_ids = tokenizer.encode(continuation, add_special_tokens=False)

    if not prefix_ids or not continuation_ids:
        raise ValueError("Prefix or continuation")

    input_ids = torch.tensor(
        [prefix_ids + continuation_ids],
        dtype=torch.long,
        device=device,
    )

    start = len(prefix_ids)

    with torch.inference_mode():
        outputs = llm_model(input_ids, use_cache=False)

        logits = outputs.logits[0, start - 1:-1].float()
        targets = input_ids[0, start:]

        token_scores = (logits.gather(1, targets[:, None]).squeeze(1) - torch.logsumexp(logits, dim=-1))

        return token_scores.sum().item()

SYSTEM_PROMPT = "A conversation between two people.\n"

def build_transition_scores(texts, speakers, alpha=0.75):
    ids = sorted(texts)

    lines = {
        i: f"{'A' if speakers[i] == 0 else 'B'}: {texts[i]}\n"
        for i in ids
    }

    neutral_scores = {
        j: continuation_logprob(SYSTEM_PROMPT, lines[j])
        for j in ids
    }

    transitions = {}

    for i in ids:
        for j in ids:
            if speakers[i] == speakers[j]:
                continue

            conditional = continuation_logprob(SYSTEM_PROMPT + lines[i], lines[j])

            transitions[i, j] = (conditional - alpha * neutral_scores[j])

    return transitions

In [20]:
# Beam Search

def beam_order(chunk_ids, first, second, speakers, transitions, beam_width=200, top_k=8):
    beam_width = max(beam_width, top_k)
    chunk_ids = sorted(chunk_ids)

    bits = {
        chunk_id: 1 << position
        for position, chunk_id in enumerate(chunk_ids)
    }

    initial_mask = bits[first] | bits[second]

    beam = [(0.0, (first, second), initial_mask)]

    for position in range(2, len(chunk_ids)):
        expected_speakers = position % 2
        merged = {}

        for score, sequence, mask in beam:
            last = sequence[-1]

            for candidate in chunk_ids:
                if mask & bits[candidate]:
                    continue
                if speakers[candidate] != expected_speakers:
                    continue

                new_mask = mask | bits[candidate]
                new_score = score + transitions[last, candidate]
                new_sequence = sequence + (candidate,)

                key = (new_mask, candidate)

                bucket = merged.setdefault(key, [])
                bucket.append((new_score, new_sequence, new_mask))
                bucket.sort(key=lambda x: x[0], reverse=True)
                del bucket[top_k:]

        beam = sorted(
            (state for bucket in merged.values() for state in bucket),
            key=lambda x: x[0],
            reverse=True,
        )[:beam_width]

    return [
        list(sequence)
        for _, sequence, _ in beam[:top_k]
    ]

In [16]:
# Main Helper Functions

def full_dialogue_score(sequence, texts, speakers):
    lines = [
        f"{'A' if speakers[i] == 0 else 'B'}: {texts[i]}\n"
        for i in sequence
    ]

    if len(lines) <= 2:
        return 0.0

    return continuation_logprob(SYSTEM_PROMPT + "".join(lines[:2]), "".join(lines[2:]))

def reconstruct_order(transcripts, first, second, speakers, alpha=0.75, beam_width=200, top_k=8):
    texts = {
        item["id"]: item["text"]
        for item in transcripts
    }

    transitions = build_transition_scores(texts, speakers, alpha)

    candidates = beam_order(texts.keys(), first, second, speakers, transitions, beam_width, top_k)

    reranked = [{
        "sequence": sequence,
        "score": full_dialogue_score(sequence, texts, speakers,),
    } for sequence in candidates]

    reranked.sort(
        key=lambda x: x["score"],
        reverse=True,
    )

    return reranked[0]["sequence"], reranked

In [21]:
# Final Writings

answers = {}
start_time = time.perf_counter()

for did, (first, second) in prefixes.items():
    folder = Path(TEST_DIR) / did

    files = sorted(folder.glob("chunk_*.wav"), key=lambda p: int(p.stem.split("_")[1]),)

    n = len(files)

    transcripts = []

    for file in files:
        chunk_id = int(file.stem.split("_")[1])

        text = trans_audio(str(file))

        transcripts.append({
            "id": chunk_id,
            "text": text,
        })

    speakers = identify_speakers(files, first, second)

    sequence, reranked = reconstruct_order(transcripts, first, second, speakers=speakers)

    order = [0] * n

    for pos, chunk_id in enumerate(sequence):
        order[chunk_id] = pos

    answers[did] = order

with open(OUTPUT, "w") as f:
    json.dump(answers, f)

print("Finished")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

In [24]:
# Eval

def score_one(prediction, target):
    n = len(target)

    if (
        not isinstance(prediction, list)
        or len(prediction) != n
        or not all(type(x) is int for x in prediction)
        or sorted(prediction) != list(range(n))
    ):
        return 0.0

    if n < 2:
        return 1.0

    correct = sum(
        (prediction[i] < prediction[j]) == (target[i] < target[j])
         for i in range(n) for j in range(i + 1, n)
    )

    return correct / (n * (n - 1) // 2)

# truth_path = Path(TEST_DIR) / "answers.json" # on real IOAI
truth_path = Path("hf-data/public/pretest_answers.json")

if truth_path.exists():
    targets = json.loads(truth_path.read_text())
    predictions = json.loads(Path(OUTPUT).read_text())

    scores = [
        score_one(predictions.get(did), target)
        for did, target in targets.items()
    ]

    print(f"Pairwise accuracy: {np.mean(scores):.6f}")
else:
    print('No truth')

Pairwise accuracy: 0.840089


In [25]:
# Not necessary - Eval on LeaderBoard

def evaluate_leaderboard(split):
    split_dir = Path("hf-data/private") / split
    split_prefixes = json.loads(
        (split_dir / "prefix.json").read_text()
    )

    predictions = {}
    started = time.perf_counter()

    for number, (did, (first, second)) in enumerate(
        split_prefixes.items(), start=1
    ):
        files = sorted(
            (split_dir / did).glob("chunk_*.wav"),
            key=lambda p: int(p.stem.split("_")[1]),
        )
        n = len(files)

        transcripts = [
            {
                "id": int(file.stem.split("_")[1]),
                "text": trans_audio(str(file)),
            }
            for file in files
        ]

        speakers = identify_speakers(files, first, second)
        sequence, _ = reconstruct_order(
            transcripts, first, second, speakers=speakers
        )

        assert sorted(sequence) == list(range(n)), did
        assert sequence[:2] == [first, second], did

        ranks = [0] * n
        for position, chunk_id in enumerate(sequence):
            ranks[chunk_id] = position

        predictions[did] = ranks
        print(
            f"{split}: {number}/{len(split_prefixes)}",
            flush=True,
        )

    output_path = Path(f"answers_{split}.json")
    output_path.write_text(
        json.dumps(predictions),
        encoding="utf-8",
    )

    targets = json.loads(
        (split_dir.parent / f"{split}_answers.json").read_text()
    )
    assert set(predictions) == set(targets)

    accuracy = np.mean([
        score_one(predictions[did], target)
        for did, target in targets.items()
    ])

    print(f"\n{split}")
    print(f"Pairwise accuracy: {accuracy:.6f}")
    print(f"Elapsed: {time.perf_counter() - started:.1f}s")
    print(f"Saved: {output_path}")

    return float(accuracy)

score_a = evaluate_leaderboard("test_leaderboard_a")
score_b = evaluate_leaderboard("test_leaderboard_b")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

test_leaderboard_a: 1/100


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

test_leaderboard_a: 2/100


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

test_leaderboard_a: 3/100


... (392 outputs left out)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

test_leaderboard_b: 99/100


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

test_leaderboard_b: 100/100

test_leaderboard_b
Pairwise accuracy: 0.782086
Elapsed: 203.6s
Saved: answers_test_leaderboard_b.json


In [ ]:
# And the total points score you would get is about
# score = 100 * ((your solution on leaderboard b, ours is about 0.782) - (Base solution, about 0.4905)) / (Upper Benchmark (here it is 0.85) - (Base solution))
# so, score = 81 points, impressive, isn't it? :)